In [4]:
import numpy as np 
import matplotlib.pyplot as plt


#data 
x = np.array([1,5,9,13,17,19,22,25,27,28,29,33,37,41,42,43,45,49,54,56], dtype=float)
y = np.array([1.7,0.1,2.3,2.7,1.7,5.0,3.0,1.5,3.5,1.8,
              1.3,2.5,3.1,-5.1,-28.1,33.8,6.3,-1.6,2.4,3.3])


def thomas_algorithm(a, b, c, d):
    n = len(b)
    # Forward elimination
    for i in range(1, n):
        w = a[i-1] / b[i-1]
        b[i] = b[i] - w * c[i-1]
        d[i] = d[i] - w * d[i-1]
    # Back substitution
    x = np.zeros(n)
    x[-1] = d[-1] / b[-1]
    for i in range(n-2, -1, -1):
        x[i] = (d[i] - c[i] * x[i+1]) / b[i]
    return x


def build_spline_M(x, y):
    n = len(x)
    h = np.diff(x)
    A = np.zeros(n-2)   # sub-diagonal
    B = np.zeros(n-2)   # main diagonal
    C = np.zeros(n-2)   # super-diagonal
    D = np.zeros(n-2)   # right-hand side
 
    for idx in range(size):
        i = idx + 1
        A[idx] = h[i-1]
        B[idx] = 2 * (h[i-1] + h[i])
        C[idx] = h[i]
        D[idx] = 6 * ((y[i+1] - y[i]) / h[i] - (y[i] - y[i-1]) / h[i-1])
 
    M_interior = thomas_algorithm(A, B, C, D)
 
    # Natural spline boundary conditions: M[0] = M[-1] = 0
    M = np.zeros(n)
    M[1:-1] = M_interior
    return M


def setup_tridiagonal_matrix(x, y):
    n = len(x)
    h = np.diff(x) # Step sizes between data points
    # Initialize the tridiagonal matrix coefficients
    A = np.zeros(n - 2) # Subdiagonal
    B = np.zeros(n - 2) # Main diagonal
    C = np.zeros(n - 2) # Superdiagonal
    D = np.zeros(n - 2) # Right-hand side vector
    # Populate the tridiagonal matrix coefficients
    for i in range(1, n - 1):
        A[i + 1] = h[i-1] # Subdiagonal
        B[i + 1] = 2 * (h[i-1] + h[i]) # Main diagonal
        C[i + 1] = h[i] # Superdiagonal
        D[i + 1] = 6 * ((y[i+1] - y[i]) / h[i] - (y[i] - y[i-1]) / h[i-1]) # Right-hand side vector
    return A, B, C, D

#cubic spline interpolation 
def evaluate_spline(x_data, y_data, M, x_query):
    n = len(x_data)
    h = np.diff(x_data) # step sizes
    i = np.searchsorted(x_data, x_query, side='right') - 1
    i = np.clip(i, 0, n - 2) # find interval
    hi = h[i]
    dx_left = x_query - x_data[i] # (x - xᵢ)
    dx_right = x_data[i+1] - x_query # (xᵢ₊₁ - x)
    S = (
        (M[i] / (6*hi)) * dx_right**3 + # Term 1
        (M[i+1] / (6*hi)) * dx_left**3 + # Term 2
        (y_data[i] /hi - M[i] *hi/6) * dx_right + # Term 3
        (y_data[i+1] /hi - M[i+1] *hi/6) * dx_left # Term 4
    )
    return S

#least squares 
def leastSquares(X, Y):
    n = len(X)
    sum_x = np.sum(X)
    sum_y = np.sum(Y)
    sum_xy = np.sum(X * Y)
    sum_x2 = np.sum(X ** 2)

    m = (n*sum_xy - sum_x*sum_y) / (n*sum_x2 - sum_x**2)
    c = (sum_y - m*sum_x) / n

    return m, c


#polynomial fit
def build_vandermonde(x, degree):
    V = np.zeros((len(x), degree + 1))
    for j in range(degree + 1):
        V[:, j] = x ** j
    return V
 
def poly_fit(x, y, degree):
    V   = build_vandermonde(x, degree)
    VtV = V.T @ V
    Vty = V.T @ y
    cond = np.linalg.cond(VtV)
    print(f"  Condition number of VtV (degree {degree}): {cond:.4e}")
    coeffs = np.linalg.solve(VtV, Vty)
    return coeffs
 
def eval_poly(coeffs, x_eval):
    V = build_vandermonde(x_eval, len(coeffs) - 1)
    return V @ coeffs



#cubic spline plot
A, B, C, D = setup_tridiagonal_matrix(x, y)
M = build_spline_M(x,y)
x_query = thomas_algorithim(A, B, C, D)
s = evaluate_spline(x, y, M, x_query)
x_spline = np.linspace(x[0], x[-1], 500)

plt.figure()
plt.plot(x_spline, s, color = 'red', label = 'cubic spline')
plt.scatter(x,y, color ='blue', label='data points')
plt.xlabel('quarters')
plt.ylabel('GDP growth rate')
plt.legend()
plt.show


#polynmial fit 
coeffs = poly_fit(x, y, degree=4)
x_poly = np.linspace(x[0], x[-1], 500)
y_poly = eval_poly(coeffs, x_plot)
y_poly_data = eval_poly(coeffs, x)
residuals_poly = y - y_poly_data


#spline vs polynomial plot
plt.figure()
plt.plot(x_spline, s, color = 'red', label = 'cubic spline')
plt.plot(x_plot, y_poly, color ='blue', label = 'polynomial')
plt.scatter(x, y, color ='yellow', label = 'data points')
plt.title('cubic spline vs polynomial')
plt.xlabel('quarters')
plt.ylabel('GDP growth rate')
plt.legend()
plt.show()


#residuals 
plt.figure()
plt.stem(x, residuals_poly, color = 'black')
plt.axhline(0, color='blue', linestyle='--')
plt.title('polynomial residuals')
plt.xlabel('quarters')
plt.ylabel('residuals')
plt.show()


#excluding covid years
covid_x = {41, 42, 43, 45}
new = np.array([xi not in covid_x for xi in x])
x_covid = x[new]
y_covid = y[new]
slope, intercept = leastSquares(x_covid, y_covid)
y_linear_plot    = slope * x_poly + intercept
y_linear_at_data = slope * x_covid + intercept
residuals_linear = y_covid - y_linear_at_data

plt.figure()
plt.plot(x_poly,y_linear_plot, colore='red', label = 'linear fit')
plt.scatter(x_covid, y_covid, color='blue', label ='non-covid data'
plt.axhline(0, color='black', linestyle='--')
plt.title('linear least square fit (without covid years)')         
plt.xlabel('quarters')
plt.ylabel('GDP growth rate')
plt.legend()
plt.show()

IndexError: index 18 is out of bounds for axis 0 with size 18